# Day 2 · Exercise 3b: The same DC problem in four different tools

Every notebook so far used **PuLP**. This one takes the Nike India DC problem from exercise 3 and solves it four times, with four different tools. Nothing about the problem changes, only the code you write.

**What you'll be able to say by the end**
1. "PuLP is a **modeling layer**, not a solver. It writes the model down and hands it to a **solver** such as HiGHS, CBC or Gurobi."
2. "Pyomo, PuLP and OR-Tools all describe the same thing: variables, an objective and constraints. Switching tools is mostly a change of syntax."
3. "Underneath, every LP or MIP is just a **matrix**: one row per constraint, one column per variable. SciPy makes you build that matrix yourself."
4. "CP-SAT is a different kind of solver that only works with whole numbers. It's very strong on scheduling and routing."
5. "I'd prototype in PuLP or Pyomo with HiGHS. In production I'd point the same model at Gurobi if speed mattered."

_Setup check: make sure this notebook runs on the repo's `.venv`. Nothing to learn here._

In [1]:
import sys
from pathlib import Path

VENV = (Path.cwd().parent / ".venv").resolve()
assert Path(sys.prefix).resolve() == VENV, f"Wrong kernel: {sys.prefix}. Select 'supply-chain-or (.venv)'."
print("Kernel:", sys.executable)

Kernel: /Users/abhijeet/Personal/supply_chain_or/.venv/bin/python


## 1. Modeling layer vs solver: what PuLP actually is

Think of a restaurant.

- **You write the order** ("2 chairs, 1 table, no more than 100 units of wood"). That's the **modeling layer**: PuLP, Pyomo, OR-Tools, or a plain matrix in SciPy.
- **The kitchen cooks it.** That's the **solver**: HiGHS, CBC, SCIP, CP-SAT, Gurobi, CPLEX. It does the actual math (simplex, branch and bound).

Different waiters write the order slightly differently, and some kitchens are faster. The meal you get is the same.

| Tool | Modeling layer? | Solver it uses here | Typical use |
|---|---|---|---|
| **PuLP** | yes | HiGHS (PuLP 4.2's default) | Quick LP/MIP prototypes |
| **Pyomo** | yes | HiGHS, or Gurobi/CPLEX/Ipopt if installed | Big or nonlinear models, research, many solvers |
| **OR-Tools** | yes | CP-SAT (also SCIP, GLOP) | Scheduling, routing, assignment |
| **SciPy `milp`** | no, you pass matrices | HiGHS | Small problems, or when you already have the matrix |
| **gurobipy** | yes | Gurobi (commercial) | Production at large companies |

PuLP 4.2 in this repo ships **HiGHS** and **CP-SAT**, not the older CBC:

In [2]:
import importlib.util
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import pulp
import pyomo.environ as pyo
from ortools.sat.python import cp_model
from scipy.optimize import Bounds, LinearConstraint, milp

print("Solvers PuLP can use here:", pulp.listSolvers(onlyAvailable=True))

Solvers PuLP can use here: ['CPSAT', 'HiGHS']


## 2. The data (same as exercise 3)

We load the data straight from `scripts/03_facility_location.py`, so all four tools solve exactly the same problem:
- 5 candidate DC sites, each with a weekly fixed cost and a capacity
- 20 cities, each with weekly demand
- shipping cost per pair = ₹5 handling + ₹0.05 per road-km

From exercise 3 we already know the answer: open **Delhi, Mumbai, Bengaluru, Kolkata** for **₹33.8 lakh a week**. Every tool below should land on it.

In [3]:
ROOT = Path.cwd().parent
spec = importlib.util.spec_from_file_location("fl", ROOT / "scripts" / "03_facility_location.py")
fl = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fl)

SITES, CITIES = fl.SITES, fl.CITIES
demand, capacity, fixed_cost, ship_cost, lakh = fl.demand, fl.capacity, fl.fixed_cost, fl.ship_cost, fl.lakh

pd.DataFrame({"fixed cost per week": {s: lakh(v) for s, v in fixed_cost.items()},
              "capacity (pairs/week)": capacity})

,fixed cost per week,capacity (pairs/week)
Delhi,Rs 6.0 lakh,30000
Mumbai,Rs 7.0 lakh,30000
Bengaluru,Rs 5.5 lakh,25000
Kolkata,Rs 4.0 lakh,20000
Nagpur,Rs 3.0 lakh,35000


The model in words, so we can check each tool against it:

- **Decide:** `open[s]` is 1 or 0 for each site; `ship[s, c]` is pairs sent from site s to city c each week (0 or more)
- **Minimise:** fixed cost of the open sites + shipping cost
- **Rule 1 (demand):** every city receives at least what it needs
- **Rule 2 (linking):** a site ships at most its capacity × `open[s]`, so a closed site ships nothing

That's 5 + 100 = **105 variables** and 20 + 5 = **25 constraints**.

## 3. PuLP + HiGHS (what you already know)

Written out again here so you can compare it line by line with the others.

In [4]:
def solve_pulp():
    m = pulp.LpProblem("facility_location", pulp.LpMinimize)
    is_open = {s: m.add_variable(f"open_{s}", cat="Binary") for s in SITES}
    ship = {(s, c): m.add_variable(f"ship_{s}_{c}", lowBound=0) for s in SITES for c in CITIES}

    m += (pulp.lpSum(fixed_cost[s] * is_open[s] for s in SITES)
          + pulp.lpSum(ship_cost.loc[s, c] * ship[s, c] for s in SITES for c in CITIES))
    for c in CITIES:
        m += pulp.lpSum(ship[s, c] for s in SITES) >= demand[c]
    for s in SITES:
        m += pulp.lpSum(ship[s, c] for c in CITIES) <= capacity[s] * is_open[s]

    start = time.time()
    m.solve(pulp.HiGHS(msg=False))
    return {"cost": pulp.value(m.objective), "open": [s for s in SITES if is_open[s].varValue > 0.5],
            "seconds": time.time() - start}


results = {"PuLP + HiGHS": solve_pulp()}
results["PuLP + HiGHS"]

{'cost': 3380518.8319988623,
 'open': ['Delhi', 'Mumbai', 'Bengaluru', 'Kolkata'],
 'seconds': 0.012600183486938477}

## 4. Pyomo + HiGHS

Pyomo is wordier, but it's structured like the math you'd write on paper:

| On paper | Pyomo |
|---|---|
| "for every site s in SITES" | `m.SITES = pyo.Set(...)` |
| "open[s] ∈ {0, 1}" | `pyo.Var(m.SITES, domain=pyo.Binary)` |
| "for every city c: Σ ship[s,c] ≥ demand[c]" | a **rule** function, called once per city |

A **rule** is a small function that builds one constraint for one index. Pyomo calls `meet_demand(m, "Delhi")`, then `meet_demand(m, "Mumbai")`, and so on, which gives 20 constraints from one function.

The best reason to use Pyomo: the solver is a single string. Change `"highs"` to `"gurobi"` and nothing else changes. Pyomo also handles **nonlinear** models (with Ipopt), which PuLP can't.

In [5]:
def solve_pyomo(solver_name="highs"):
    m = pyo.ConcreteModel()
    m.SITES = pyo.Set(initialize=SITES)
    m.CITIES = pyo.Set(initialize=CITIES)

    m.is_open = pyo.Var(m.SITES, domain=pyo.Binary)
    m.ship = pyo.Var(m.SITES, m.CITIES, domain=pyo.NonNegativeReals)

    m.cost = pyo.Objective(
        expr=sum(fixed_cost[s] * m.is_open[s] for s in m.SITES)
        + sum(ship_cost.loc[s, c] * m.ship[s, c] for s in m.SITES for c in m.CITIES),
        sense=pyo.minimize,
    )

    def meet_demand(m, c):
        return sum(m.ship[s, c] for s in m.SITES) >= demand[c]

    def link_capacity(m, s):
        return sum(m.ship[s, c] for c in m.CITIES) <= capacity[s] * m.is_open[s]

    m.meet_demand = pyo.Constraint(m.CITIES, rule=meet_demand)
    m.link_capacity = pyo.Constraint(m.SITES, rule=link_capacity)

    start = time.time()
    pyo.SolverFactory(solver_name).solve(m)  # swap "highs" for "gurobi" or "cplex" if licensed
    return {"cost": pyo.value(m.cost), "open": [s for s in SITES if pyo.value(m.is_open[s]) > 0.5],
            "seconds": time.time() - start}


results["Pyomo + HiGHS"] = solve_pyomo()
results["Pyomo + HiGHS"]

{'cost': 3380518.831998862,
 'open': ['Delhi', 'Mumbai', 'Bengaluru', 'Kolkata'],
 'seconds': 0.01181793212890625}

## 5. OR-Tools CP-SAT: a solver that only knows whole numbers

CP-SAT is a different kind of engine. HiGHS uses LP relaxations and branch and bound, as in exercise 3. CP-SAT mixes **constraint programming** (rule out impossible values fast) with **SAT solving** (the logic engines used to check computer chips). It's very good when everything is yes/no or small integers: shift rosters, job sequencing, vehicle routes.

The catch: **constraints may only contain whole numbers**, and CP-SAT is happiest when the objective is in whole numbers too. (Recent versions accept a fractional objective, but they quietly scale it to integers behind the scenes.) So we do the scaling ourselves, where we can see it:
- measure cost in **paise** instead of rupees (₹12.34 → 1234 paise), rounded
- ship whole pairs of shoes (which makes sense anyway)

A quick check that rounding to paise can't change the answer: the largest rounding error is half a paisa per pair, and 64,500 pairs ship each week, so at most ₹323 out of ₹33.8 lakh. That's far smaller than the cost gap between any two DC networks in exercise 3.

In [6]:
def solve_cpsat():
    paise = {(s, c): round(100 * ship_cost.loc[s, c]) for s in SITES for c in CITIES}

    m = cp_model.CpModel()
    is_open = {s: m.new_bool_var(f"open_{s}") for s in SITES}
    ship = {(s, c): m.new_int_var(0, demand[c], f"ship_{s}_{c}") for s in SITES for c in CITIES}

    m.minimize(sum(100 * fixed_cost[s] * is_open[s] for s in SITES)
               + sum(paise[k] * ship[k] for k in ship))
    for c in CITIES:
        m.add(sum(ship[s, c] for s in SITES) >= demand[c])
    for s in SITES:
        m.add(sum(ship[s, c] for c in CITIES) <= capacity[s] * is_open[s])

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 30
    solver.parameters.num_workers = 8
    start = time.time()
    status = solver.solve(m)
    seconds = time.time() - start
    # Re-price the plan in exact rupees so it is comparable with the others
    cost = (sum(fixed_cost[s] * solver.value(is_open[s]) for s in SITES)
            + sum(ship_cost.loc[k] * solver.value(ship[k]) for k in ship))
    return {"cost": float(cost), "open": [s for s in SITES if solver.value(is_open[s])], "seconds": seconds,
            "status": solver.status_name(status)}


results["OR-Tools CP-SAT"] = solve_cpsat()
results["OR-Tools CP-SAT"]

{'cost': 3380518.831998862,
 'open': ['Delhi', 'Mumbai', 'Bengaluru', 'Kolkata'],
 'seconds': 0.055910348892211914,
 'status': 'OPTIMAL'}

## 6. SciPy `milp`: no modeling layer, just the matrix

This is the most useful section for understanding what the tools above actually do. PuLP, Pyomo and OR-Tools all turn your model into the same thing before the solver sees it:

> minimise **c · x**  subject to  **lo ≤ A x ≤ hi**,  with some entries of x forced to be whole numbers

- **x** is one long list of all 105 variables: `[open_Delhi, …, open_Nagpur, ship_Delhi→Delhi, ship_Delhi→Mumbai, …]`
- **c** is the cost of each variable, in the same order
- **A** has one row per constraint and one column per variable. Each cell says how much that variable counts in that rule.

Worked by hand for two rows:
- **Row "demand Kolkata":** a 1 in each column `ship_s→Kolkata` (5 of them), 0 everywhere else. Bounds: `lo = 4,500`, `hi = ∞`.
- **Row "capacity Delhi":** a 1 in each column `ship_Delhi→c` (20 of them), and **−30,000** in the column `open_Delhi`. Bounds: `lo = −∞`, `hi = 0`. That's the linking rule moved to one side: Σ ship − 30,000 × open ≤ 0.

SciPy makes you build this yourself, which is tedious and easy to get wrong. That's why modeling layers exist.

In [7]:
def build_matrix():
    n_s, n_c = len(SITES), len(CITIES)
    col_open = {s: i for i, s in enumerate(SITES)}
    col_ship = {(s, c): n_s + i * n_c + j for i, s in enumerate(SITES) for j, c in enumerate(CITIES)}
    n_vars = n_s + n_s * n_c

    c_vec = np.zeros(n_vars)
    for s in SITES:
        c_vec[col_open[s]] = fixed_cost[s]
    for (s, c), j in col_ship.items():
        c_vec[j] = ship_cost.loc[s, c]

    A = np.zeros((n_c + n_s, n_vars))
    lo, hi = np.zeros(n_c + n_s), np.zeros(n_c + n_s)
    for r, c in enumerate(CITIES):            # sum_s ship[s,c] >= demand[c]
        for s in SITES:
            A[r, col_ship[s, c]] = 1
        lo[r], hi[r] = demand[c], np.inf
    for k, s in enumerate(SITES):             # sum_c ship[s,c] - capacity[s] * open[s] <= 0
        r = n_c + k
        for c in CITIES:
            A[r, col_ship[s, c]] = 1
        A[r, col_open[s]] = -capacity[s]
        lo[r], hi[r] = -np.inf, 0

    integrality = np.zeros(n_vars)
    integrality[:n_s] = 1                                   # the "open" columns are whole numbers...
    upper = np.full(n_vars, np.inf)
    upper[:n_s] = 1                                         # ...between 0 and 1, so yes/no
    return c_vec, A, lo, hi, integrality, Bounds(0, upper), col_open, col_ship


def solve_scipy():
    c_vec, A, lo, hi, integrality, bounds, col_open, _ = build_matrix()
    start = time.time()
    res = milp(c_vec, constraints=LinearConstraint(A, lo, hi), integrality=integrality, bounds=bounds)
    return {"cost": res.fun, "open": [s for s in SITES if res.x[col_open[s]] > 0.5],
            "seconds": time.time() - start}


results["SciPy milp (HiGHS)"] = solve_scipy()
results["SciPy milp (HiGHS)"]

{'cost': 3380518.8319988614,
 'open': ['Delhi', 'Mumbai', 'Bengaluru', 'Kolkata'],
 'seconds': 0.013607025146484375}

Here's the matrix **A** as a picture. Blue cells are +1, red cells are −capacity, and blank cells are 0. Hover to see each value.

What to notice:
- **Top 20 rows (demand):** each row has 5 blue cells spread across the matrix, one per site.
- **Bottom 5 rows (capacity):** each row is a solid blue band of 20 (one site shipping to every city) plus a single red cell in the `open` column. That red cell is the big-M linking constraint from exercise 3.
- **Most of the matrix is empty.** Real supply chain models are like this too: millions of columns, but each constraint touches only a few. Solvers are built to exploit this, which is why they can handle huge models.

In [8]:
_, A, *_ = build_matrix()
cols = [f"open {s}" for s in SITES] + [f"ship {s}→{c}" for s in SITES for c in CITIES]
rows = [f"demand {c}" for c in CITIES] + [f"capacity {s}" for s in SITES]
shown = np.where(A == 0, np.nan, np.sign(A))           # blank = 0, blue = +1, red = -capacity

fig = go.Figure(go.Heatmap(
    z=shown, x=cols, y=rows, customdata=A, colorscale=[[0, "#c0392b"], [1, "#2e6fba"]],
    zmin=-1, zmax=1, showscale=False, xgap=1, ygap=1,
    hovertemplate="row: %{y}<br>column: %{x}<br>coefficient: %{customdata:,.0f}<extra></extra>",
))
fig.update_xaxes(showticklabels=False, title_text=f"{A.shape[1]} columns (5 open + 100 ship variables)")
fig.update_yaxes(autorange="reversed", tickfont_size=9)
fig.update_layout(title=f"The whole model as a {A.shape[0]} x {A.shape[1]} matrix "
                        f"({np.count_nonzero(A)} non-zero cells out of {A.size})",
                  height=520, width=1000, template="plotly_white")
fig.show()

## 7. Side by side

In [9]:
pd.DataFrame({
    name: {"open DCs": " + ".join(r["open"]), "cost": lakh(r["cost"]), "seconds": round(r["seconds"], 3)}
    for name, r in results.items()
}).T

,open DCs,cost,seconds
PuLP + HiGHS,Delhi + Mumbai + Bengaluru + Kolkata,Rs 33.8 lakh,0.013
Pyomo + HiGHS,Delhi + Mumbai + Bengaluru + Kolkata,Rs 33.8 lakh,0.012
OR-Tools CP-SAT,Delhi + Mumbai + Bengaluru + Kolkata,Rs 33.8 lakh,0.056
SciPy milp (HiGHS),Delhi + Mumbai + Bengaluru + Kolkata,Rs 33.8 lakh,0.014


All four agree: **Delhi + Mumbai + Bengaluru + Kolkata, ₹33.8 lakh a week.** On a problem this small, every tool solves in a few hundredths of a second, so speed doesn't decide anything here. What differs is how pleasant the code is to write and what each tool can grow into.

| | Write it | Good at | Watch out for |
|---|---|---|---|
| **PuLP** | Shortest, most readable | Fast prototypes of LP/MIP | Linear only; PuLP 4 changed the API from the tutorials online |
| **Pyomo** | More setup (sets, rules) | Swap solvers in one word; nonlinear models; big structured models | Slower to *build* very large models |
| **OR-Tools CP-SAT** | Short, but whole numbers only | Scheduling, routing, many yes/no choices | Must scale fractional costs to integers |
| **SciPy milp** | You build the matrix | Tiny models, no extra dependency | Error-prone; index bugs fail silently |

**Interview soundbite:** *"I modeled it in PuLP with HiGHS to prototype. The formulation carries over to any tool, since it's the same matrix underneath. For production I'd use Pyomo or gurobipy so I could switch to Gurobi for speed on the large instances. If the problem turned into routing or shift scheduling, I'd look at OR-Tools CP-SAT."*

**Try it yourself**
1. In `solve_pyomo`, change `"highs"` to `"appsi_highs"`. Same answer: it's just a different connection to the same kitchen.
2. In `build_matrix`, change one demand number in `lo` and check that only SciPy's answer moves. That's how you know the matrix *is* the model.
3. In `solve_cpsat`, change the capacity rule to `<= 0.9 * capacity[s] * is_open[s]` (run DCs at 90% at most) and read the `TypeError` CP-SAT gives you. Then fix it by rounding `0.9 * capacity[s]` to a whole number.